# D3 全部站点坐标修正（基于 Abs_PM 匹配）

修正 District 3 所有高速公路的 PeMS 站点坐标

In [1]:
import pandas as pd
import numpy as np
import json
import os
import glob
import folium
from math import radians, sin, cos, sqrt, atan2
from tqdm import tqdm

# ============== 配置 ==============

# Caltrans GeoJSON 文件路径
CALTRANS_GEOJSON = "./SHN_Postmiles_Tenth.geojson"

# PeMS 元数据目录
META_DIR = "../d03_meta"

# 输出目录
OUTPUT_DIR = "./output/d3_coordinate_correction"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("配置完成！")

配置完成！


## 1. 加载 PeMS D3 元数据（先加载以确定需要哪些高速）

In [2]:
META_COLUMNS = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

# 加载 D3 元数据
pattern = os.path.join(META_DIR, "d03_text_meta_*.txt")
files = glob.glob(pattern)
if not files:
    raise FileNotFoundError("未找到 D3 元数据文件")

meta_file = sorted(files)[-1]
print(f"使用元数据文件: {meta_file}")

pems_df = pd.read_csv(meta_file, sep='\t', names=META_COLUMNS, header=0, 
                      dtype={'ID': str, 'Fwy': str})
print(f"总记录数: {len(pems_df)}")

使用元数据文件: ../d03_meta/d03_text_meta_2025_12_30.txt
总记录数: 1903


In [3]:
# 筛选有效数据
pems_target = pems_df[
    pems_df['Latitude'].notna() & 
    pems_df['Longitude'].notna() &
    pems_df['Abs_PM'].notna() &
    pems_df['Fwy'].notna() &
    (pems_df['Latitude'] > 30) &
    (pems_df['Latitude'] < 42)
].copy()

print(f"有效站点数: {len(pems_target)}")
print(f"\n高速分布:")
print(pems_target['Fwy'].value_counts())
print(f"\n类型分布:")
print(pems_target['Type'].value_counts())

有效站点数: 1901

高速分布:
Fwy
80     480
50     466
99     334
5      285
51      90
65      65
20      49
70      24
113     18
89      18
12      16
49      12
160      9
28       6
45       6
162      6
244      5
267      4
193      3
275      2
505      2
16       1
Name: count, dtype: int64

类型分布:
Type
ML    883
OR    430
HV    281
FR    279
FF     28
Name: count, dtype: int64


In [4]:
# 获取需要加载的高速列表
target_routes = pems_target['Fwy'].unique().tolist()
# 转为整数
target_routes_int = []
for r in target_routes:
    try:
        target_routes_int.append(int(r))
    except:
        pass

print(f"需要加载的高速: {sorted(target_routes_int)}")
print(f"共 {len(target_routes_int)} 条高速")

需要加载的高速: [5, 12, 16, 20, 28, 45, 49, 50, 51, 65, 70, 80, 89, 99, 113, 160, 162, 193, 244, 267, 275, 505]
共 22 条高速


## 2. 加载 Caltrans 官方数据

In [5]:
print(f"加载 Caltrans 数据: {CALTRANS_GEOJSON}")
print("文件较大，请稍候...")

with open(CALTRANS_GEOJSON, 'r') as f:
    data = json.load(f)

print(f"总特征数: {len(data['features'])}")

加载 Caltrans 数据: ./SHN_Postmiles_Tenth.geojson
文件较大，请稍候...
总特征数: 311198


In [6]:
# 只提取目标高速
records = []
for feature in tqdm(data['features'], desc="筛选 Caltrans 数据"):
    props = feature.get('properties', {})
    route = props.get('Route')
    
    # 只加载 D3 需要的高速
    if route not in target_routes_int:
        continue
    
    geom = feature.get('geometry', {})
    coords = geom.get('coordinates', [None, None])
    
    odometer = props.get('Odometer')
    if odometer is None or coords[0] is None:
        continue
    
    records.append({
        'Route': route,
        'County': props.get('County'),
        'PM': props.get('PM'),
        'Odometer': odometer,
        'AlignCode': props.get('AlignCode'),
        'Direction': props.get('Direction'),
        'Longitude': coords[0],
        'Latitude': coords[1],
    })

caltrans_df = pd.DataFrame(records)
print(f"\nCaltrans 有效里程桩数: {len(caltrans_df)}")
print(f"\n各高速里程桩数量:")
print(caltrans_df['Route'].value_counts())

筛选 Caltrans 数据: 100%|██████████| 311198/311198 [00:00<00:00, 808530.83it/s]



Caltrans 有效里程桩数: 64846

各高速里程桩数量:
Route
5      16280
99      8600
49      6146
89      4990
20      4480
80      4252
70      3684
12      2412
162     2272
50      2262
65      1956
16      1674
45      1446
113     1254
160     1050
193      722
505      670
267      238
28       224
51       188
244       28
275       18
Name: count, dtype: int64


## 3. 坐标修正

In [7]:
def calc_distance_feet(lat1, lon1, lat2, lon2):
    """计算两点距离（英尺）"""
    R = 3958.8 * 5280
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c


def correct_coordinates_by_abspm(pems_df, caltrans_df):
    """
    根据 Abs_PM 匹配 Caltrans Odometer，修正 PeMS 坐标
    """
    results = []
    
    # 预先按 Route 和 AlignCode 分组，加速查找
    caltrans_grouped = {}
    for route in caltrans_df['Route'].unique():
        route_data = caltrans_df[caltrans_df['Route'] == route]
        caltrans_grouped[route] = {
            'right': route_data[route_data['AlignCode'].isin(['Right', 'Right Side'])],
            'left': route_data[route_data['AlignCode'].isin(['Left', 'Left Side'])],
        }
    
    for _, row in tqdm(pems_df.iterrows(), total=len(pems_df), desc="修正坐标"):
        result = row.to_dict()
        result['Original_Lat'] = row['Latitude']
        result['Original_Lon'] = row['Longitude']
        
        # 获取路线号
        try:
            route = int(row['Fwy'])
        except:
            result['Corrected_Lat'] = None
            result['Corrected_Lon'] = None
            result['Status'] = 'invalid_fwy'
            results.append(result)
            continue
        
        # 检查是否有该路线的 Caltrans 数据
        if route not in caltrans_grouped:
            result['Corrected_Lat'] = None
            result['Corrected_Lon'] = None
            result['Status'] = 'no_caltrans_route'
            results.append(result)
            continue
        
        # 方向映射: N/E -> Right, S/W -> Left
        if row['Dir'] in ['N', 'E']:
            caltrans_subset = caltrans_grouped[route]['right']
        else:
            caltrans_subset = caltrans_grouped[route]['left']
        
        if len(caltrans_subset) == 0:
            result['Corrected_Lat'] = None
            result['Corrected_Lon'] = None
            result['Status'] = 'no_caltrans_direction'
            results.append(result)
            continue
        
        # 找 Odometer 最接近 Abs_PM 的点
        pm_diff = (caltrans_subset['Odometer'] - row['Abs_PM']).abs()
        nearest_idx = pm_diff.idxmin()
        nearest = caltrans_subset.loc[nearest_idx]
        
        # 计算修正距离
        correction_dist = calc_distance_feet(
            row['Latitude'], row['Longitude'],
            nearest['Latitude'], nearest['Longitude']
        )
        
        result['Corrected_Lat'] = nearest['Latitude']
        result['Corrected_Lon'] = nearest['Longitude']
        result['Match_Odometer'] = nearest['Odometer']
        result['Match_AlignCode'] = nearest['AlignCode']
        result['PM_Diff'] = abs(row['Abs_PM'] - nearest['Odometer'])
        result['Correction_Dist_ft'] = correction_dist
        result['Status'] = 'corrected'
        
        results.append(result)
    
    return pd.DataFrame(results)


print("修正函数定义完成")

修正函数定义完成


In [8]:
# 执行修正
pems_corrected = correct_coordinates_by_abspm(pems_target, caltrans_df)

print("\n修正状态统计:")
print(pems_corrected['Status'].value_counts())

修正坐标: 100%|██████████| 1901/1901 [00:00<00:00, 2203.84it/s]


修正状态统计:
Status
corrected    1901
Name: count, dtype: int64


In [9]:
# 修正距离统计
corrected = pems_corrected[pems_corrected['Status'] == 'corrected']

print(f"成功修正站点数: {len(corrected)}")
print("\n修正距离统计 (英尺):")
print(f"  平均: {corrected['Correction_Dist_ft'].mean():.1f}")
print(f"  中位数: {corrected['Correction_Dist_ft'].median():.1f}")
print(f"  最小: {corrected['Correction_Dist_ft'].min():.1f}")
print(f"  最大: {corrected['Correction_Dist_ft'].max():.1f}")

print("\n修正距离分布:")
bins = [(0, 50), (50, 100), (100, 200), (200, 500), (500, 1000), (1000, float('inf'))]
for low, high in bins:
    count = ((corrected['Correction_Dist_ft'] >= low) & (corrected['Correction_Dist_ft'] < high)).sum()
    label = f"{low}-{high}" if high != float('inf') else f">{low}"
    pct = count / len(corrected) * 100
    print(f"  {label} ft: {count} 个站点 ({pct:.1f}%)")

成功修正站点数: 1901

修正距离统计 (英尺):
  平均: 1249.8
  中位数: 232.7
  最小: 2.4
  最大: 42466.4

修正距离分布:
  0-50 ft: 240 个站点 (12.6%)
  50-100 ft: 222 个站点 (11.7%)
  100-200 ft: 378 个站点 (19.9%)
  200-500 ft: 441 个站点 (23.2%)
  500-1000 ft: 192 个站点 (10.1%)
  >1000 ft: 428 个站点 (22.5%)


In [10]:
# 按高速统计修正距离
print("各高速平均修正距离:")
fwy_stats = corrected.groupby('Fwy')['Correction_Dist_ft'].agg(['mean', 'median', 'max', 'count'])
fwy_stats.columns = ['平均(ft)', '中位数(ft)', '最大(ft)', '站点数']
fwy_stats = fwy_stats.round(1).sort_values('站点数', ascending=False)
print(fwy_stats.head(20))

各高速平均修正距离:
      平均(ft)  中位数(ft)   最大(ft)  站点数
Fwy                                
80     609.7    124.1  17089.2  480
50     176.5    141.1   2781.7  466
99    1163.7    503.0   6390.8  334
5     1154.3   1162.2   1537.0  285
51     188.1    127.5   1165.7   90
65     134.7    132.4    315.1   65
20   13166.8  13347.7  14360.0   49
70    1502.6    805.5   4185.4   24
89    2797.7   1350.7   8394.9   18
113    170.1     97.8    657.6   18
12     153.2    133.9    280.2   16
49   22107.7  25529.7  27760.1   12
160   6524.0   6782.3   7069.9    9
162  10841.0   8839.4  23539.5    6
45      51.7     56.1     67.4    6
28      63.3     62.8     81.9    6
244    154.5    178.2    266.6    5
267    126.6    126.6    215.1    4
193  42456.2  42461.9  42466.4    3
275    777.1    777.1    780.4    2


In [11]:
# 显示修正距离最大的站点
print("修正距离最大的 20 个站点:")
top20 = corrected.nlargest(20, 'Correction_Dist_ft')[
    ['ID', 'Fwy', 'Dir', 'Type', 'Abs_PM', 'Match_Odometer', 'PM_Diff', 'Correction_Dist_ft']
].copy()
top20['Correction_Dist_ft'] = top20['Correction_Dist_ft'].round(1)
top20['PM_Diff'] = top20['PM_Diff'].round(3)
print(top20.to_string(index=False))

修正距离最大的 20 个站点:
     ID Fwy Dir Type  Abs_PM  Match_Odometer  PM_Diff  Correction_Dist_ft
3038098 193   E   FR  10.071       10.035000    0.036             42466.4
3038095 193   E   OR  10.072       10.035000    0.037             42461.9
3038093 193   W   OR  10.077       10.035000    0.042             42440.4
3423026  49   N   FR 196.681      196.660004    0.021             27760.1
3423027  49   S   OR 196.739      196.759995    0.021             27584.7
3423024  49   S   ML 196.789      196.759995    0.029             27482.6
3423021  49   N   ML 196.789      196.759995    0.029             27440.7
3423025  49   N   OR 196.792      196.759995    0.032             27434.5
3423028  49   S   FR 196.818      196.860001    0.042             27218.8
3423013  49   N   FR 197.424      197.460007    0.036             23840.5
3423015  49   S   OR 197.456      197.460007    0.004             23750.7
3069011 162   E   ML  72.578       72.605003    0.027             23539.5
3069012 162   W   ML  

In [12]:
# 保存完整修正结果
output_file = os.path.join(OUTPUT_DIR, 'pems_d3_corrected_full.csv')
pems_corrected.to_csv(output_file, index=False)
print(f"已保存完整结果: {output_file}")

已保存完整结果: ./output/d3_coordinate_correction/pems_d3_corrected_full.csv


## 4. 导出修正后的元数据

In [13]:
# 生成可用于构图的元数据
pems_for_graph = pems_corrected[pems_corrected['Status'] == 'corrected'].copy()

# 使用修正后的坐标
pems_for_graph['Latitude'] = pems_for_graph['Corrected_Lat']
pems_for_graph['Longitude'] = pems_for_graph['Corrected_Lon']

# 只保留构图需要的列
graph_columns = ['ID', 'Fwy', 'Dir', 'District', 'County', 'City',
                 'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
                 'Type', 'Lanes', 'Name']

available_cols = [c for c in graph_columns if c in pems_for_graph.columns]
pems_export = pems_for_graph[available_cols].copy()

# 保存
export_file = os.path.join(OUTPUT_DIR, 'pems_d3_meta_corrected.csv')
pems_export.to_csv(export_file, index=False)
print(f"已保存修正后元数据: {export_file}")
print(f"站点数: {len(pems_export)}")

已保存修正后元数据: ./output/d3_coordinate_correction/pems_d3_meta_corrected.csv
站点数: 1901


In [14]:
# 按高速分别保存
for fwy in pems_export['Fwy'].unique():
    fwy_data = pems_export[pems_export['Fwy'] == fwy]
    fwy_file = os.path.join(OUTPUT_DIR, f'pems_{fwy}_meta_corrected.csv')
    fwy_data.to_csv(fwy_file, index=False)
    print(f"  {fwy}: {len(fwy_data)} 站点 -> {fwy_file}")

  50: 466 站点 -> ./output/d3_coordinate_correction/pems_50_meta_corrected.csv
  5: 285 站点 -> ./output/d3_coordinate_correction/pems_5_meta_corrected.csv
  99: 334 站点 -> ./output/d3_coordinate_correction/pems_99_meta_corrected.csv
  80: 480 站点 -> ./output/d3_coordinate_correction/pems_80_meta_corrected.csv
  51: 90 站点 -> ./output/d3_coordinate_correction/pems_51_meta_corrected.csv
  160: 9 站点 -> ./output/d3_coordinate_correction/pems_160_meta_corrected.csv
  70: 24 站点 -> ./output/d3_coordinate_correction/pems_70_meta_corrected.csv
  244: 5 站点 -> ./output/d3_coordinate_correction/pems_244_meta_corrected.csv
  275: 2 站点 -> ./output/d3_coordinate_correction/pems_275_meta_corrected.csv
  65: 65 站点 -> ./output/d3_coordinate_correction/pems_65_meta_corrected.csv
  113: 18 站点 -> ./output/d3_coordinate_correction/pems_113_meta_corrected.csv
  89: 18 站点 -> ./output/d3_coordinate_correction/pems_89_meta_corrected.csv
  28: 6 站点 -> ./output/d3_coordinate_correction/pems_28_meta_corrected.csv
  267:

## 5. 可视化

In [15]:
def create_correction_map(pems_corrected, output_path, title="D3 坐标修正"):
    """
    创建修正前后对比地图
    """
    corrected = pems_corrected[pems_corrected['Status'] == 'corrected']
    
    center_lat = corrected['Original_Lat'].mean()
    center_lon = corrected['Original_Lon'].mean()
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles=None)
    
    # 底图
    folium.TileLayer('OpenStreetMap', name='OSM').add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
        attr='Google', name='Google 街道'
    ).add_to(m)
    folium.TileLayer(
        tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr='Google', name='Google 混合'
    ).add_to(m)
    
    # 原始位置（红色）
    original_group = folium.FeatureGroup(name='原始 PeMS 坐标')
    for _, row in corrected.iterrows():
        folium.CircleMarker(
            [row['Original_Lat'], row['Original_Lon']],
            radius=5,
            color='#F44336',
            fill=True,
            fillOpacity=0.7,
            weight=1,
            popup=f"原始<br>{row['ID']}<br>Fwy {row['Fwy']}{row['Dir']}",
            tooltip=f"{row['ID']}"
        ).add_to(original_group)
    original_group.add_to(m)
    
    # 修正后位置（绿色）
    corrected_group = folium.FeatureGroup(name='修正后坐标')
    for _, row in corrected.iterrows():
        folium.CircleMarker(
            [row['Corrected_Lat'], row['Corrected_Lon']],
            radius=5,
            color='#4CAF50',
            fill=True,
            fillOpacity=0.7,
            weight=1,
            popup=f"修正<br>{row['ID']}<br>偏移 {row['Correction_Dist_ft']:.0f}ft",
            tooltip=f"{row['ID']} ({row['Correction_Dist_ft']:.0f}ft)"
        ).add_to(corrected_group)
    corrected_group.add_to(m)
    
    # 修正连线（只显示偏移 > 50ft 的）
    lines_group = folium.FeatureGroup(name='修正连线 (>50ft)')
    for _, row in corrected.iterrows():
        if row['Correction_Dist_ft'] < 50:
            continue
        
        if row['Correction_Dist_ft'] < 200:
            color = '#FFC107'
        elif row['Correction_Dist_ft'] < 500:
            color = '#FF9800'
        else:
            color = '#F44336'
        
        folium.PolyLine(
            [[row['Original_Lat'], row['Original_Lon']],
             [row['Corrected_Lat'], row['Corrected_Lon']]],
            color=color,
            weight=2,
            dash_array='5,5',
            opacity=0.8
        ).add_to(lines_group)
    lines_group.add_to(m)
    
    # 图例
    legend = f"""
    <div style="position:fixed; bottom:50px; left:50px; z-index:1000;
                background:white; padding:12px; border:2px solid #333; border-radius:5px;">
        <div style="font-weight:bold; margin-bottom:8px;">{title}</div>
        <div><span style="color:#F44336;">●</span> 原始坐标</div>
        <div><span style="color:#4CAF50;">●</span> 修正后坐标</div>
        <div style="margin-top:5px;"><b>连线颜色:</b></div>
        <div><span style="color:#FFC107;">—</span> 50-200 ft</div>
        <div><span style="color:#FF9800;">—</span> 200-500 ft</div>
        <div><span style="color:#F44336;">—</span> >500 ft</div>
        <div style="margin-top:5px; font-size:10px;">站点数: {len(corrected)}</div>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend))
    
    folium.LayerControl().add_to(m)
    m.save(output_path)
    print(f"地图已保存: {output_path}")
    return m


print("可视化函数定义完成")

可视化函数定义完成


In [16]:
# 生成 D3 全局地图
d3_map = create_correction_map(
    pems_corrected,
    os.path.join(OUTPUT_DIR, 'correction_d3_all.html'),
    title="D3 全部站点坐标修正"
)
d3_map

地图已保存: ./output/d3_coordinate_correction/correction_d3_all.html


In [17]:
# 为主要高速生成单独的地图
main_highways = ['5', '50', '80', '99']

for fwy in main_highways:
    fwy_data = pems_corrected[pems_corrected['Fwy'] == fwy]
    if len(fwy_data) > 0:
        create_correction_map(
            fwy_data,
            os.path.join(OUTPUT_DIR, f'correction_{fwy}.html'),
            title=f"Fwy {fwy} 坐标修正"
        )

地图已保存: ./output/d3_coordinate_correction/correction_5.html
地图已保存: ./output/d3_coordinate_correction/correction_50.html
地图已保存: ./output/d3_coordinate_correction/correction_80.html
地图已保存: ./output/d3_coordinate_correction/correction_99.html


## 总结

### 输出文件

| 文件 | 说明 |
|------|------|
| `pems_d3_corrected_full.csv` | 完整修正结果 |
| `pems_d3_meta_corrected.csv` | 修正后元数据（用于构图）|
| `pems_{Fwy}_meta_corrected.csv` | 各高速单独的修正元数据 |
| `correction_d3_all.html` | D3 全局对比地图 |
| `correction_{Fwy}.html` | 各高速单独的对比地图 |

### 在分层构图中使用

```python
# 加载修正后的 D3 元数据
meta_df = pd.read_csv('./output/d3_coordinate_correction/pems_d3_meta_corrected.csv')

# 或加载单个高速
meta_99 = pd.read_csv('./output/d3_coordinate_correction/pems_99_meta_corrected.csv')
```